# Suivi de la Déforestation et des Perturbations Forestières par IA

## Introduction
Le bassin du Congo abrite la deuxième plus grande forêt tropicale du monde. Ce notebook utilise l'intelligence artificielle pour surveiller en temps réel la déforestation en détectant des anomalies croisées entre la perte de biomasse optique et les caractéristiques structurales de l'IA.

## Objectifs
*   **Alerte précoce** : Identifier les coupes rases ou sélectives récentes.
*   **Validation structurelle** : Utiliser l'IA Prithvi pour confirmer que la forêt a été altérée mécaniquement.
*   **Quantification de perte** : Estimer l'aire déforestée sur la période d'étude.

## Méthodologie
1.  **Setup** : Installation de TerraTorch.
2.  **Acquisition** : Images Sentinel-2 filtrées.
3.  **Encodage Prithvi** : Transformation du signal en vecteurs IA.
4.  **Alerte Hybride** : Calcul croisé des anomalies de texture (IA) et de l'indice NDVI.

In [ ]:
# ====================================================
# ÉTAPE 1 : Configuration
# ====================================================
!pip install geemap earthengine-api rasterio terratorch torch matplotlib -q

import ee, geemap, torch, rasterio
import numpy as np
import matplotlib.pyplot as plt
from terratorch import BACKBONE_REGISTRY

try: ee.Initialize()
except: 
    ee.Authenticate()
    ee.Initialize(project='geocongoai-api')

print('✅ Système prêt')

## Zone d'Étude (ROI)
Alignement sur la zone régionale d'étude au Congo.

In [ ]:
# ====================================================
# ÉTAPE 2 : Définition de la ROI
# ====================================================
roi = ee.Geometry.Rectangle([15.0, -5.0, 16.0, -4.0])

Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(roi, 8)
Map.addLayer(roi, {'color': 'red'}, 'Zone d'étude')
Map

## Acquisition des Données
Nous exportons les 6 bandes principales pour une analyse IA robuste.

In [ ]:
# ====================================================
# ÉTAPE 3 : Données Sentinel-2
# ====================================================
image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
         .filterBounds(roi).median().clip(roi))

geemap.ee_export_image(image.select(['B2','B3','B4','B8','B11','B12']), 'forest.tif', scale=30, region=roi)

## Inférence et Alerte Déforestation
Nous calculons l'écart type des caractéristiques IA. Combiné à un NDVI bas, ce signal devient une alerte de déforestation très fiable.

In [ ]:
# ====================================================
# ÉTAPE 4 : Inférence et Algorithme d'Alerte
# ====================================================
model = BACKBONE_REGISTRY.build('prithvi_eo_v2_300', num_frames=1, in_chans=6, pretrained=True).eval().to('cpu')

with rasterio.open('forest.tif') as src: img_file = src.read().astype(np.float32) / 10000.0
with torch.no_grad():
    out = model(torch.from_numpy(img_file).unsqueeze(0))
    feats = out[0] if isinstance(out, list) else out
    
feats_np = feats[0, 1:].numpy()
side = int(np.sqrt(feats_np.shape[0]))
deviation = np.std(feats_np, axis=1).reshape(side, -1)

red, nir = img_file[2], img_file[3]
ndvi = (nir - red) / (nir + red + 1e-8)

alert = deviation * (1.0 - (ndvi + 1.0)/2.0)

plt.figure(figsize=(10, 8))
plt.imshow(alert, cmap='Reds')
plt.colorbar(label='Indice d'alerte déforestation IA')
plt.title('Alertes de Perturbation de la Couverture Forestière')
plt.axis('off')
plt.show()